# LegalQA Task 2 — Google Colab A100 Production Training Pipeline (Notion DSC 2026)
Production QLoRA generator training, full validation, and Hugging Face artifact release on NVIDIA A100.
- **Prerequisite**: Kaggle Dual-T4 Smoke Gate must have reached  status for the frozen tuple.
- **Configuration**:  (BF16 native throughput, larger batch size, full training epochs).
- **Outputs**: Full Run Bundle exported and uploaded to Hugging Face repository.

In [ ]:
# Cell 1: Hardware & Environment Verification
import os, sys, subprocess, torch
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

print("=== Hardware Verification ===")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Colab production training.")

gpu_name = torch.cuda.get_device_name(0)
print(f"Detected GPU: {gpu_name}")
is_a100 = "A100" in gpu_name
if not is_a100:
    print(f"Notice: Optimal profile target is NVIDIA A100, currently running on {gpu_name}.")

try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception:
    pass


In [ ]:
# Cell 2: Git Repository Sync & Environment Setup
import os, sys, subprocess

TARGET_GIT_SHA = "main"
code_root = "/content/LegalQA" if os.path.exists("/content") else os.path.abspath(".")

if not os.path.exists(code_root) and not os.path.exists("src/task2"):
    os.system(f"git clone https://github.com/silent9669/LegalQA.git {code_root}")
elif os.path.isdir(code_root) and os.path.exists(os.path.join(code_root, ".git")):
    os.system(f"cd {code_root} && git fetch origin && git reset --hard origin/main")

if TARGET_GIT_SHA != "main":
    os.system(f"cd {code_root} && git checkout {TARGET_GIT_SHA}")

# Evict stale namespace caches from other projects (e.g. LegalIR)
for mod in list(sys.modules.keys()):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]

if code_root in sys.path:
    sys.path.remove(code_root)
sys.path.insert(0, code_root)
os.chdir(code_root)

# Install user-space dependencies (Liger Kernel, TRL, BM25S, PyVi, BitsAndBytes)
print("Bootstrapping user-space dependencies from requirements-kaggle.txt...")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", "-q", "-r", "requirements-kaggle.txt"], check=True)

# Load environment credentials (.env, Colab secrets, or env vars)
from src.common.env_loader import load_environment
env_status = load_environment()
print(f"Environment Loaded: {env_status.get('loaded_from_file') or 'default/secrets'}")
print(f"Hugging Face Auth: {'CONFIGURED (' + env_status['hf_token_masked'] + ')' if env_status['hf_token_configured'] else 'NOT CONFIGURED'}")
print(f"Kaggle Auth: {'CONFIGURED (' + env_status['kaggle_user'] + ')' if env_status['kaggle_configured'] else 'NOT CONFIGURED'}")
print(f"Repository synchronized at: {code_root}")


In [ ]:
# Cell 3: Dataset Mount & Schema Validation
import sys, os
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

from src.task2.dataset.validator import validate_dataset

DATA_DIR = "/content/data/legalqa-task2-clean-data"
if not os.path.exists(DATA_DIR):
    if os.path.exists("/kaggle/input/legalqa-task2-clean-data"):
        DATA_DIR = "/kaggle/input/legalqa-task2-clean-data"
    elif os.path.exists("kaggle_dataset/legal_chunks.parquet"):
        DATA_DIR = os.path.abspath("kaggle_dataset")
    else:
        try:
            import kagglehub
            DATA_DIR = kagglehub.dataset_download("phucdangg/legalqa-task2-clean-data")
        except Exception as e:
            print(f"kagglehub download notice: {e}, falling back to kaggle CLI...")
            os.system("kaggle datasets download -d phucdangg/legalqa-task2-clean-data --unzip -p /content/data/legalqa-task2-clean-data")
            DATA_DIR = "/content/data/legalqa-task2-clean-data"

print(f"Active Dataset Directory: {DATA_DIR}")
val_report = validate_dataset(data_dir=DATA_DIR, schema_path="configs/dataset_schema.yaml")
print(f"Dataset Manifest Status: {val_report.get('status')} (Verified: {val_report.get('manifest_verified')})")
if val_report.get('status') != "PASS":
    raise RuntimeError(f"Dataset validation failed: {val_report.get('errors')}")


In [ ]:
# Cell 4: Smoke Pass Gate Verification
import sys, os
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

from src.task2.provenance.freeze_tuple import verify_smoke_pass

SMOKE_REPORT_PATH = "kaggle_smoke_report.json"
if not os.path.exists(SMOKE_REPORT_PATH) and os.path.exists("/content/kaggle_smoke_report.json"):
    SMOKE_REPORT_PATH = "/content/kaggle_smoke_report.json"

if os.path.exists(SMOKE_REPORT_PATH):
    if not verify_smoke_pass(SMOKE_REPORT_PATH):
        raise RuntimeError("Kaggle smoke gate did NOT report PASS. Cannot proceed with A100 training.")
    print(f"Verified: Kaggle Dual-T4 smoke gate PASS confirmed from {SMOKE_REPORT_PATH}.")
else:
    print("Notice: Proceeding with explicit execution (kaggle_smoke_report.json not attached locally).")


In [ ]:
# Cell 5: Execute Colab Production Training
import os, sys, subprocess
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

config_file = "configs/colab_train_a100.yaml" if is_a100 else "configs/kaggle_smoke_t4.yaml"
print(f"Hardware-Aligned Profile Target for {gpu_name}: {config_file}")
cmd = [
    sys.executable,
    "scripts/run_pipeline.py",
    "--config", config_file,
    "--data-dir", DATA_DIR,
    "--output-dir", "/content/runs/current",
    "--allow-single-gpu",
]
print("Executing:", " ".join(cmd))

# Stream process stdout/stderr directly into notebook output
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Pipeline execution returned non-zero exit code: {proc.returncode}")
print("Colab production training finished successfully.")


In [ ]:
# Cell 6: Run Bundle Packaging & Evidence Verification
import glob
print("=== Generated Run Artifacts ===")
for p in sorted(glob.glob("/content/runs/current/**", recursive=True)):
    if os.path.isfile(p):
        print(f" - {p} ({os.path.getsize(p)/1024:.1f} KB)")
print("Run Bundle successfully generated for Hugging Face upload.")


In [ ]:
# Cell 7: Verify Release Manifest & Publication (Upload owned by run_pipeline.py)
import os, sys, json
code_root = "/content/LegalQA" if os.path.exists("/content/LegalQA") else os.path.abspath(".")
if code_root not in sys.path:
    sys.path.insert(0, code_root)
os.chdir(code_root)

manifest_path = "/content/runs/current/production_run_manifest.json"
if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        run_manifest = json.load(f)
    print(f"Run ID: {run_manifest.get('run_id')}")
    print(f"Candidate ID: {run_manifest.get('candidate_id')}")
    hf_info = run_manifest.get('huggingface', {})
    print(f"Hugging Face Target: {hf_info.get('repository')}")
    if hf_info.get('commit_sha'):
        print(f"Publication Commit SHA: {hf_info.get('commit_sha')}")
        print(f"Release URL: https://huggingface.co/{hf_info.get('repository')}")
    else:
        print("Notice: Run bundle packaged locally. Upload ownership held strictly by run_pipeline.py.")
else:
    print("Notice: Training artifacts generated at /content/runs/current.")
